In [1]:
import numpy as np
from tensorflow.keras.layers import Dense, Input, Flatten
from tensorflow.keras.models import Model

# Ruta a tu archivo de pesos generado
ruta_pesos = "c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/Glucose_prediction_2026/models/cSigP01_Linear_H4_Fold1.weights.h5"

# 1. Recrear la arquitectura exacta del script
history_length = 8  # Cambia esto si usaste otro valor en tu ejecución
i = Input(shape=(history_length,))
x = Flatten()(i)
x = Dense(1, activation=None)(x)
model = Model(inputs=[i], outputs=[x])

# 2. Cargar los pesos entrenados
model.load_weights(ruta_pesos)

# 3. La capa Dense es la última capa (índice -1)
# get_weights() devuelve una lista: [matriz_de_pesos, array_de_sesgos]
pesos, sesgo = model.layers[-1].get_weights()

# Aplanar los pesos para que sea un simple array 1D
w = pesos.flatten()
b = sesgo[0]

print("Coeficientes (w):", w)
print("Sesgo (b):", b)

ValueError: Layer count mismatch when loading weights from file. Model expected 1 layers, found 0 saved layers.

In [2]:
import h5py
import numpy as np

# Ruta a tu archivo de pesos generado
ruta_pesos = "c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/Glucose_prediction_2026/models/cSigP01_Linear_H4_Fold1.weights.h5"

w = None
b = None

with h5py.File(ruta_pesos, 'r') as f:
    # Función que recorre todas las carpetas internas del archivo H5
    def encontrar_parametros(name, obj):
        global w, b
        if isinstance(obj, h5py.Dataset):
            # Keras suele llamar 'kernel' a los multiplicadores (pesos) y 'bias' al sesgo
            if 'kernel' in name.lower() or 'weight' in name.lower():
                w = np.array(obj)
            elif 'bias' in name.lower():
                b = np.array(obj)
                
    # Ejecutamos la búsqueda
    f.visititems(encontrar_parametros)

if w is not None and b is not None:
    # Aplanamos los arrays para tener listas simples
    w = w.flatten()
    b = b.flatten()[0]
    
    print("¡Pesos extraídos con éxito!\n")
    print("Coeficientes (W):", list(np.round(w, 4)))
    print("Sesgo (B):", round(b, 4))
else:
    print("No se encontraron los pesos con los nombres estándar. Estructura interna del archivo:")
    # Si falla por algún motivo de formato extraño, esto imprimirá dónde están escondidos
    def imprimir_estructura(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f" -> Dataset encontrado: {name} | Forma: {obj.shape}")
    with h5py.File(ruta_pesos, 'r') as f:
        f.visititems(imprimir_estructura)

No se encontraron los pesos con los nombres estándar. Estructura interna del archivo:
 -> Dataset encontrado: layers/dense/vars/0 | Forma: (8, 1)
 -> Dataset encontrado: layers/dense/vars/1 | Forma: (1,)
 -> Dataset encontrado: metrics/mean/vars/0 | Forma: ()
 -> Dataset encontrado: metrics/mean/vars/1 | Forma: ()
 -> Dataset encontrado: metrics/mean_metric_wrapper/vars/0 | Forma: ()
 -> Dataset encontrado: metrics/mean_metric_wrapper/vars/1 | Forma: ()
 -> Dataset encontrado: metrics/root_mean_squared_error/vars/0 | Forma: ()
 -> Dataset encontrado: metrics/root_mean_squared_error/vars/1 | Forma: ()
 -> Dataset encontrado: optimizer/vars/0 | Forma: ()
 -> Dataset encontrado: optimizer/vars/1 | Forma: (8, 1)
 -> Dataset encontrado: optimizer/vars/2 | Forma: (8, 1)
 -> Dataset encontrado: optimizer/vars/3 | Forma: (1,)
 -> Dataset encontrado: optimizer/vars/4 | Forma: (1,)


In [3]:
import h5py
import numpy as np

# Cambia esto por la ruta real a tu archivo
ruta_pesos = "c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/Glucose_prediction_2026/models/cSigP01_Linear_H4_Fold1.weights.h5" 

with h5py.File(ruta_pesos, 'r') as f:
    # Accedemos directamente a las rutas que descubrimos
    w = np.array(f['layers/dense/vars/0']).flatten()
    b = np.array(f['layers/dense/vars/1']).flatten()[0]

print("¡Pesos extraídos con éxito!\n")
# Imprimimos los valores listos para copiar y pegar
print(f"W = {list(w)}")
print(f"B = {b}")

¡Pesos extraídos con éxito!

W = [0.043323018, -0.042692296, -0.06836648, 0.29572073, -0.87264186, 1.5070925, -2.7938073, 2.8550856]
B = 6.68784761428833


In [4]:
import h5py
import numpy as np

# Cambia esto por la ruta real a tu archivo
ruta_pesos = "c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/Glucose_prediction_2026/models/cSigP01_Linear_H4_Fold1.weights.h5" 

try:
    with h5py.File(ruta_pesos, 'r') as f:
        # Extraemos los pesos y el sesgo directamente de la estructura del H5
        W = np.array(f['layers/dense/vars/0']).flatten()
        B = np.array(f['layers/dense/vars/1']).flatten()[0]

    # Construir la ecuación en formato texto
    ecuacion = "Prediccion = "
    
    for i, w in enumerate(W):
        # Generamos las partes de la ecuación: (peso * x1) + (peso * x2) ...
        # Usamos x1, x2... para representar el historial de glucosa
        ecuacion += f"({w:.6f} * x{i+1}) + "
    
    # Añadimos el sesgo (bias) al final
    ecuacion += f"{B:.6f}"

    print("¡Pesos extraídos con éxito del archivo H5!\n")
    print("La ecuación lineal de tu predictor es:")
    print("-" * 80)
    print(ecuacion)
    print("-" * 80)
    print("\nNota: 'x1' es la lectura más antigua de tu ventana histórica y 'x8' es la más reciente.")

except KeyError:
    print("Error: No se encontraron las variables en la ruta esperada dentro del archivo H5.")
except FileNotFoundError:
    print(f"Error: No se encontró el archivo en la ruta '{ruta_pesos}'.")
except Exception as e:
    print(f"Ocurrió un error inesperado: {e}")

¡Pesos extraídos con éxito del archivo H5!

La ecuación lineal de tu predictor es:
--------------------------------------------------------------------------------
Prediccion = (0.043323 * x1) + (-0.042692 * x2) + (-0.068366 * x3) + (0.295721 * x4) + (-0.872642 * x5) + (1.507092 * x6) + (-2.793807 * x7) + (2.855086 * x8) + 6.687848
--------------------------------------------------------------------------------

Nota: 'x1' es la lectura más antigua de tu ventana histórica y 'x8' es la más reciente.
